# Notebook 04a — Phase 2: MobileFaceNet **Fine-tune** on Bollywood Faces

**Phase:** 2 (Indian demographic specialization, fine-tune branch)  
**Purpose:** Take InsightFace's pretrained `w600k_mbf` MobileFaceNet (WebFace600K + ArcFace) and fine-tune it on the 100-identity Bollywood Faces dataset. Produces `mobilefacenet_bollywood_ft.tflite`.  
**Expected runtime:** ~40–80 minutes (depending on ONNX→Keras conversion success)  
**GPU required:** **YES — T4 ×2**  
**Run in parallel with:** Notebook 04b (from-scratch baseline on the second Kaggle account)

## Strategy

1. Pull InsightFace `buffalo_s.zip` → extract `w600k_mbf.onnx` (same as Notebook 02).
2. **Convert ONNX → trainable Keras model** via `onnx2tf` SavedModel output. Try multiple loaders in cascade — if one fails the next is attempted. This is the brittle step.
3. Use MediaPipe to pre-crop faces from the 12.4k raw images (cached to `/kaggle/working/crops/`).
4. Add an ArcFace classification head (100 classes, margin=0.5, scale=64).
5. Fine-tune at **low LR** (1e-4) — the pretrained weights are good; we just nudge them toward Indian-demographic features. High LR would catastrophically forget WebFace600K.
6. **EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)** + **ModelCheckpoint(save_best_only=True)** + ReduceLROnPlateau.
7. Drop the ArcFace head, export the backbone-only model → TFLite.

## What to send back to Claude

Paste the literal output of cells 5, 7, 9, 11, 13, 15, 17, 19, 23, 25, 27 (discovery, gathering, cropping, split, pipeline, backbone load, training model, training, TFLite conversion, sanity, summary). If anything fails, paste the **full traceback** of the failing cell.

## What to do after the run

Download `/kaggle/working/models/mobilefacenet_bollywood_ft.tflite` and the JSON history + curves PNG → `kaggle_downloads/04a_mobilefacenet_ft/` on the laptop. Notebook 05 will compare this against `mobilefacenet_bollywood_scratch.tflite` (from Notebook 04b) and the original `mobilefacenet.tflite` (InsightFace baseline) via pair verification, then ship the EER winner.

## 1 — Config

Lower LR than the from-scratch notebook (1e-4 vs 1e-3) — we're nudging pretrained weights, not learning from zero.

In [ ]:
STRATEGY               = "fine_tune"
NUM_CLASSES            = 100
EPOCHS_MAX             = 20
BATCH_SIZE             = 64
IMAGE_SIZE             = (112, 112)
EMBEDDING_DIM          = 512
LR                     = 1e-4              # low for fine-tune
ARCFACE_SCALE          = 64.0
ARCFACE_MARGIN         = 0.5
VAL_SPLIT              = 0.1               # per-celeb
SEED                   = 42
TFLITE_FILENAME        = "mobilefacenet_bollywood_ft.tflite"

WORK_DIR    = "/kaggle/working"
MODELS_OUT  = f"{WORK_DIR}/models"
REPORTS_OUT = f"{WORK_DIR}/reports"
CROPS_DIR   = f"{WORK_DIR}/crops"

import os
for d in (MODELS_OUT, REPORTS_OUT, CROPS_DIR):
    os.makedirs(d, exist_ok=True)
print(f"Strategy: {STRATEGY}")
print(f"Config locked. LR={LR}, epochs_max={EPOCHS_MAX}, batch={BATCH_SIZE}")

## 2 — Clone repo + install deps\n\nInstall: `onnx2tf` + `onnx` (ONNX→Keras conversion), `sklearn` (metrics). Face cropping uses OpenCV's pre-installed Haar cascade — no `mediapipe` because pip-installing it on Kaggle is fragile (numpy ABI mismatches + API namespace drift between versions; both failure modes hit us on first run).

In [ ]:
import subprocess, sys

REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

print("\nInstalling deps (onnx2tf, onnx, sklearn — no mediapipe)…")
!pip install -q onnx2tf onnx onnxruntime onnx_graphsurgeon sng4onnx scikit-learn

## 3 — Discover Bollywood dataset

Three sub-folders (`bollywood_celeb_faces_0`, `bollywood_celeb_faces_1`, `bollywood_celeb_faces2`) under the dataset root, each with `<Celebrity_Name>/<n>.jpg` structure.

In [ ]:
import glob

print("/kaggle/input contents:")
if os.path.isdir("/kaggle/input"):
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  {d}")

DATA_ROOT_CANDIDATES = [
    "/kaggle/input/100-bollywood-celebrity-faces",
    "/kaggle/input/datasets/havingfun/100-bollywood-celebrity-faces",
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if os.path.isdir(c)), None)
if DATA_ROOT is None:
    # Glob fallback
    hits = glob.glob("/kaggle/input/**/bollywood_celeb_faces*", recursive=True)
    if hits:
        # Take the parent of the first hit
        DATA_ROOT = os.path.dirname(hits[0])
assert DATA_ROOT, "Could not locate Bollywood dataset — paste /kaggle/input/ tree to Claude"
print(f"\nDATA_ROOT: {DATA_ROOT}")

# Find the 3 split folders
SPLIT_FOLDERS = sorted([os.path.join(DATA_ROOT, d) for d in os.listdir(DATA_ROOT)
                       if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.startswith("bollywood_celeb_faces")])
print(f"\nSplit folders ({len(SPLIT_FOLDERS)}):")
for f in SPLIT_FOLDERS:
    n_celebs = len([d for d in os.listdir(f) if os.path.isdir(os.path.join(f, d))])
    print(f"  {f}  ({n_celebs} celebs)")

## 4 — Walk dataset, gather images per celebrity

Merge images across the 3 split folders by celebrity name. Same celeb in multiple split folders → all images combined.

In [ ]:
from collections import defaultdict

IMG_EXTS = (".jpg", ".jpeg", ".png")
celeb_to_imgs = defaultdict(list)

for split_folder in SPLIT_FOLDERS:
    for celeb_dir in sorted(os.listdir(split_folder)):
        celeb_path = os.path.join(split_folder, celeb_dir)
        if not os.path.isdir(celeb_path):
            continue
        for f in os.listdir(celeb_path):
            if f.lower().endswith(IMG_EXTS):
                celeb_to_imgs[celeb_dir].append(os.path.join(celeb_path, f))

celebs = sorted(celeb_to_imgs.keys())
celeb_to_idx = {c: i for i, c in enumerate(celebs)}

counts = [len(celeb_to_imgs[c]) for c in celebs]
print(f"Total celebs: {len(celebs)}")
print(f"Total images: {sum(counts)}")
print(f"Min/max/median per celeb: {min(counts)}/{max(counts)}/{sorted(counts)[len(counts)//2]}")
print(f"\nTop 5 (most images):  " + ", ".join(f"{c} ({len(celeb_to_imgs[c])})" for c in sorted(celebs, key=lambda c: -len(celeb_to_imgs[c]))[:5]))
print(f"Bottom 5 (fewest):    " + ", ".join(f"{c} ({len(celeb_to_imgs[c])})" for c in sorted(celebs, key=lambda c: len(celeb_to_imgs[c]))[:5]))

if NUM_CLASSES != len(celebs):
    print(f"\nNOTE: NUM_CLASSES in config ({NUM_CLASSES}) != actual celeb count ({len(celebs)}). Adjusting.")
    NUM_CLASSES = len(celebs)

## 5 — Pre-compute face crops with OpenCV Haar cascade\n\nDetect the largest face per image (OpenCV's Haar cascade, pre-installed with `cv2`), crop with 20% margin, resize to 112×112, save as JPG to `/kaggle/working/crops/<celeb>/<n>.jpg`. Done **once**, then training reads from the cache.\n\nImages where Haar finds no face fall back to a centre crop. Haar is less accurate than MediaPipe BlazeFace (~5–10% lower detection rate) but for celebrity portrait photos the difference is small, and it has zero install overhead — critical after MediaPipe's numpy-ABI and namespace failures on first run.

In [ ]:
import cv2
import numpy as np
from tqdm.auto import tqdm

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)
assert not face_cascade.empty(), f"Failed to load Haar cascade at {CASCADE_PATH}"
print(f"Haar cascade loaded: {CASCADE_PATH}")

def detect_and_crop(img_path, out_size=112, margin=0.2):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = img_rgb.shape[:2]
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=4, minSize=(20, 20))
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
        mx = int(fw * margin); my = int(fh * margin)
        x0 = max(0, x - mx); y0 = max(0, y - my)
        x1 = min(w, x + fw + mx); y1 = min(h, y + fh + my)
        face = img_rgb[y0:y1, x0:x1]
        used_detection = True
    else:
        s = min(h, w)
        face = img_rgb[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]
        used_detection = False
    return cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_LINEAR), used_detection

n_detected = n_fallback = n_failed = 0
crop_paths_per_celeb = defaultdict(list)

all_pairs = [(c, p) for c in celebs for p in celeb_to_imgs[c]]
for celeb, src in tqdm(all_pairs, desc="cropping"):
    out_dir = os.path.join(CROPS_DIR, celeb)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(src))[0] + ".jpg")
    if os.path.isfile(out_path):
        crop_paths_per_celeb[celeb].append(out_path)
        continue
    result = detect_and_crop(src)
    if result is None:
        n_failed += 1
        continue
    face_rgb, used = result
    cv2.imwrite(out_path, cv2.cvtColor(face_rgb, cv2.COLOR_RGB2BGR))
    crop_paths_per_celeb[celeb].append(out_path)
    n_detected += int(used)
    n_fallback += int(not used)

total_crops = sum(len(v) for v in crop_paths_per_celeb.values())
print(f"\nCropping complete:")
print(f"  Total crops produced:    {total_crops}")
print(f"  Haar cascade detections: {n_detected}")
print(f"  Centre-crop fallback:    {n_fallback}")
print(f"  Failed (skipped):        {n_failed}")
print(f"  Detection rate:          {n_detected / max(1, n_detected + n_fallback):.1%}")

## 6 — Train/val split (stratified per celebrity)

Per-celebrity 90/10 split. Ensures every celebrity appears in both train and val sets.

In [ ]:
import random

rng = random.Random(SEED)
train_files, train_labels = [], []
val_files,   val_labels   = [], []
val_files_per_celeb = {}

for celeb in celebs:
    imgs = crop_paths_per_celeb[celeb][:]
    rng.shuffle(imgs)
    n_val = max(1, int(len(imgs) * VAL_SPLIT))
    val_part = imgs[:n_val]; train_part = imgs[n_val:]
    label = celeb_to_idx[celeb]
    train_files.extend(train_part); train_labels.extend([label] * len(train_part))
    val_files.extend(val_part);     val_labels.extend([label] * len(val_part))
    val_files_per_celeb[celeb] = val_part

print(f"Train: {len(train_files)}  ({len(train_labels)} labels)")
print(f"Val:   {len(val_files)}    ({len(val_labels)} labels)")
print(f"Per-celeb val min/max: {min(len(v) for v in val_files_per_celeb.values())}/{max(len(v) for v in val_files_per_celeb.values())}")

## 7 — `tf.data` pipeline

- Read pre-cropped JPG
- Cast and **normalize to `[-1, 1]`** (InsightFace's expected input range; matches contract)
- `.cache()` → entire 12k-image cropped set fits in RAM as float32 (~450 MB)
- Pre-shuffle (files, labels) Python-side (Phase 1 lesson)
- Per-epoch shuffle buffer 8192
- Augment: hflip + brightness + contrast (no spatial — would break face recognition)
- Yield `((image, label), label)` — outer label is loss target, inner feeds ArcFace head

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def decode_normalize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    img = (tf.cast(img, tf.float32) - 127.5) / 127.5   # [-1, 1]
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img, label

def make_ds(files, labels, training):
    combined = list(zip(files, labels))
    random.Random(SEED + (0 if training else 1)).shuffle(combined)
    files, labels = [list(x) for x in zip(*combined)]
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(decode_normalize, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(buffer_size=8192, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    # Reformat: yield ((image, label), label)
    ds = ds.map(lambda img, lbl: ((img, lbl), lbl), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files,   val_labels,   training=False)

# Smoke-test
for (img_b, lbl_b), lbl_t in train_ds.take(1):
    print(f"image batch: shape={img_b.shape} range=[{tf.reduce_min(img_b):.3f}, {tf.reduce_max(img_b):.3f}]")
    print(f"label batch: shape={lbl_b.shape} sample={lbl_b[:8].numpy().tolist()}")
    print(f"target:      shape={lbl_t.shape}")
    print(f"unique classes in this batch: {len(set(lbl_b.numpy().tolist()))}")
print(f"\nGPU(s): {tf.config.list_physical_devices('GPU') or 'NONE — switch accelerator to T4 ×2'}")

## 8 — Load pretrained MobileFaceNet backbone

**The brittle step.** Pull InsightFace `buffalo_s.zip`, extract `w600k_mbf.onnx`, convert via `onnx2tf` to a SavedModel directory, and try multiple loader strategies until one yields a trainable Keras model.

If all loaders fail, the cell exits with a clear error and you switch to running Notebook 04b instead. **Notebook 04b's from-scratch approach is the safety net.**

In [ ]:
import urllib.request, zipfile, shutil

BUFFALO_ZIP = f"{WORK_DIR}/buffalo_s.zip"
BUFFALO_DIR = f"{WORK_DIR}/buffalo_s"
ONNX_PATH = None

if not os.path.isfile(BUFFALO_ZIP):
    print("Downloading buffalo_s.zip from InsightFace…")
    urllib.request.urlretrieve(
        "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip",
        BUFFALO_ZIP,
    )

if not os.path.isdir(BUFFALO_DIR):
    os.makedirs(BUFFALO_DIR, exist_ok=True)
    with zipfile.ZipFile(BUFFALO_ZIP) as z:
        z.extractall(BUFFALO_DIR)

for root, _, files in os.walk(BUFFALO_DIR):
    for f in files:
        if f == "w600k_mbf.onnx":
            ONNX_PATH = os.path.join(root, f)
assert ONNX_PATH, "w600k_mbf.onnx not found"
print(f"ONNX: {ONNX_PATH}")

# Convert ONNX -> SavedModel via onnx2tf (default produces SavedModel + TFLite)
OUT_DIR = f"{WORK_DIR}/mfn_onnx2tf"
shutil.rmtree(OUT_DIR, ignore_errors=True)
!onnx2tf -i {ONNX_PATH} -o {OUT_DIR} -b 1 -ois input.1:1,3,112,112

print("\nonnx2tf produced:")
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    print(f"  {'[DIR] ' if os.path.isdir(p) else '      '}{f}")

In [ ]:
# Multi-strategy loader for the pretrained backbone
import keras
from tensorflow.keras import layers as L

backbone = None
load_method = None

# Find any SavedModel-shaped directory
candidate_dirs = []
for entry in os.listdir(OUT_DIR):
    p = os.path.join(OUT_DIR, entry)
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "saved_model.pb")):
        candidate_dirs.append(p)
# Also try OUT_DIR itself as a SavedModel
if os.path.isfile(os.path.join(OUT_DIR, "saved_model.pb")):
    candidate_dirs.append(OUT_DIR)

print(f"SavedModel candidates: {candidate_dirs}")

# Strategy 1: tf.keras.models.load_model on SavedModel
for sm in candidate_dirs:
    if backbone is not None:
        break
    try:
        m = tf.keras.models.load_model(sm)
        # Wrap if needed (already a Keras Model)
        backbone = m
        load_method = f"tf.keras.models.load_model({sm})"
        print(f"Strategy 1 succeeded with: {sm}")
    except Exception as e:
        print(f"Strategy 1 failed on {sm}: {type(e).__name__}: {e}")

# Strategy 2: tf.saved_model.load + wrap as Keras model
if backbone is None:
    for sm in candidate_dirs:
        try:
            loaded = tf.saved_model.load(sm)
            infer = loaded.signatures["serving_default"]
            # Find input/output names
            input_name = list(infer.structured_input_signature[1].keys())[0]
            output_name = list(infer.structured_outputs.keys())[0]
            print(f"  signature in: {input_name}, out: {output_name}")
            
            class PretrainedBackbone(tf.keras.layers.Layer):
                def __init__(self, infer_fn, in_name, out_name, **kwargs):
                    super().__init__(**kwargs)
                    self._infer = infer_fn
                    self._in = in_name
                    self._out = out_name
                def call(self, x):
                    return self._infer(**{self._in: x})[self._out]
            
            inp = L.Input(shape=(112, 112, 3))
            out = PretrainedBackbone(infer, input_name, output_name)(inp)
            backbone = tf.keras.Model(inp, out, name="mfn_pretrained_wrapped")
            load_method = f"tf.saved_model.load + Layer wrap ({sm})"
            print(f"Strategy 2 succeeded with: {sm}")
            print("WARNING: Strategy 2 backbone weights may not be exposed as trainable_variables.")
            print("  Fine-tuning will train ONLY the ArcFace head. The backbone stays frozen.")
            print("  This is still useful (linear probe), but less powerful than full fine-tune.")
            break
        except Exception as e:
            print(f"Strategy 2 failed on {sm}: {type(e).__name__}: {e}")

assert backbone is not None, (
    "Could not load InsightFace MobileFaceNet as a Keras model. "
    "Switch to Notebook 04b (from-scratch baseline) on this Kaggle account."
)

print(f"\n=== Backbone loaded via: {load_method} ===")
print(f"Total params: {backbone.count_params():,}")
trainable = sum(int(np.prod(v.shape)) for v in backbone.trainable_variables)
print(f"Trainable params: {trainable:,}")
print(f"Backbone input  shape: {backbone.input_shape}")
print(f"Backbone output shape: {backbone.output_shape}")

# Forward sanity
test_in = tf.random.uniform((1, 112, 112, 3), -1.0, 1.0)
test_out = backbone(test_in)
print(f"\nSanity forward: input {test_in.shape} -> output {test_out.shape}")
print(f"Output norm: {float(tf.norm(test_out)):.3f}  (should be > 0, ~5-15 for non-normalized embeddings)")

## 9 — ArcFace head + training model

Standard ArcFace formulation: L2-normalize embedding and class kernel, compute cos(θ), apply additive angular margin to the target class, scale.

The training model takes `(image, label)` as input — the label feeds the ArcFace head (so we know which class needs the margin), and is also the cross-entropy target.

In [ ]:
class ArcFaceHead(tf.keras.layers.Layer):
    def __init__(self, num_classes, scale=64.0, margin=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
    def build(self, input_shape):
        # input_shape is [(B, emb), (B,)]
        emb_dim = input_shape[0][-1]
        self.kernel = self.add_weight(
            name="kernel", shape=(emb_dim, self.num_classes),
            initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        embeddings, labels = inputs
        eps = 1e-9
        e_n = embeddings / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(embeddings), axis=1, keepdims=True)) + eps)
        k_n = self.kernel    / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(self.kernel),    axis=0, keepdims=True)) + eps)
        cos_t = keras.ops.matmul(e_n, k_n)
        cos_t = keras.ops.clip(cos_t, -1.0 + 1e-7, 1.0 - 1e-7)
        sin_t = keras.ops.sqrt(1.0 - keras.ops.square(cos_t))
        cm = keras.ops.cos(self.margin); sm = keras.ops.sin(self.margin)
        cos_tm = cos_t * cm - sin_t * sm
        mask = keras.ops.one_hot(keras.ops.cast(labels, "int32"), self.num_classes)
        logits = keras.ops.where(mask > 0.5, cos_tm, cos_t)
        return logits * self.scale
    def get_config(self):
        return {**super().get_config(),
                "num_classes": self.num_classes,
                "scale": self.scale, "margin": self.margin}

img_input   = L.Input(shape=(112, 112, 3), name="image")
label_input = L.Input(shape=(), dtype="int32", name="label")

embeddings = backbone(img_input)
logits     = ArcFaceHead(num_classes=NUM_CLASSES,
                         scale=ARCFACE_SCALE,
                         margin=ARCFACE_MARGIN, name="arcface")([embeddings, label_input])

training_model = tf.keras.Model([img_input, label_input], logits, name="training_model")

training_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"],
)
training_model.summary(line_length=120)

## 10 — Train (early stopping + best-only checkpointing)

Up to 20 epochs. EarlyStopping kills it once val_loss stops improving for 3 epochs (and restores best-epoch weights). ReduceLROnPlateau halves LR if val_loss plateaus.

In [ ]:
import json

BEST_CKPT = f"{WORK_DIR}/best_finetune.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=BEST_CKPT, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_MAX,
    callbacks=callbacks,
    verbose=2,
)

# Persist history
HIST_PATH = f"{REPORTS_OUT}/mobilefacenet_ft_training_history.json"
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
hist_data["strategy"]      = STRATEGY
hist_data["epochs_actual"] = len(history.history.get("loss", []))
hist_data["epochs_max"]    = EPOCHS_MAX
hist_data["lr"]            = LR
hist_data["num_classes"]   = NUM_CLASSES
hist_data["best_val_loss"] = float(min(history.history.get("val_loss", [float("inf")])))
with open(HIST_PATH, "w") as f:
    json.dump(hist_data, f, indent=2)
print(f"\nHistory saved -> {HIST_PATH}")

## 11 — Export backbone as TFLite

Drop the ArcFace head. The deployed model is image → backbone → 512-D raw embedding. The mobile side L2-normalizes before cosine distance (per the contract — see `shared_contracts/README.md`).

In [ ]:
# After EarlyStopping with restore_best_weights, backbone now carries the best epoch's weights.
# Build a clean inference graph: image -> backbone -> raw embedding
inf_inp = L.Input(shape=(112, 112, 3))
inf_out = backbone(inf_inp)
inference_model = tf.keras.Model(inf_inp, inf_out, name="mfn_ft_inference")

TFLITE_PATH = f"{MODELS_OUT}/{TFLITE_FILENAME}"
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
tflite_bytes = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)
print(f"TFLite saved -> {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)")

# Verify shapes
it = tf.lite.Interpreter(model_path=TFLITE_PATH)
it.allocate_tensors()
ish = it.get_input_details()[0]["shape"].tolist()
osh = it.get_output_details()[0]["shape"].tolist()
print(f"Input:  {ish}"); print(f"Output: {osh}")
assert ish == [1, 112, 112, 3], f"input shape mismatch: {ish}"
assert osh == [1, 512],        f"output shape mismatch: {osh}"

## 12 — Sanity check on held-out val celebs

For 3 randomly chosen celebs, compute pairwise cosine distance between their val-set images.

- Same-celeb pairs should have **low** cosine distance (< 0.5 ideal, < 0.7 acceptable)
- Different-celeb pairs should have **higher** cosine distance (> 0.7 ideal)

If same-celeb distance is high or different-celeb is low, the model didn't learn useful separations and we keep the InsightFace baseline.

In [ ]:
rng2 = random.Random(SEED + 100)
test_celebs = rng2.sample([c for c in celebs if len(val_files_per_celeb[c]) >= 2], 3)

def tflite_embed(it, img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112)).astype(np.float32)
    img = (img - 127.5) / 127.5
    img = img[None].astype(it.get_input_details()[0]["dtype"])
    it.set_tensor(it.get_input_details()[0]["index"], img)
    it.invoke()
    e = it.get_tensor(it.get_output_details()[0]["index"])[0]
    return e / (np.linalg.norm(e) + 1e-9)

it = tf.lite.Interpreter(model_path=TFLITE_PATH)
it.allocate_tensors()

embs = {c: [tflite_embed(it, p) for p in val_files_per_celeb[c][:3]] for c in test_celebs}

print("Same-celeb pairwise cosine distances (lower = more similar; aim < 0.5):")
for c, es in embs.items():
    if len(es) < 2: continue
    for i in range(len(es)):
        for j in range(i+1, len(es)):
            d = 1.0 - float(np.dot(es[i], es[j]))
            print(f"  {c:30s} pair ({i},{j}): {d:.4f}")

print("\nCross-celeb pairwise cosine distances (aim > 0.7):")
for i, c1 in enumerate(test_celebs):
    for j, c2 in enumerate(test_celebs):
        if i >= j: continue
        for e1 in embs[c1][:1]:
            for e2 in embs[c2][:1]:
                d = 1.0 - float(np.dot(e1, e2))
                print(f"  {c1[:20]:20s} vs {c2[:20]:20s}: {d:.4f}")

## 13 — Plots + final summary

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, ylabel) in zip(axes, [("loss", "Loss"), ("sparse_categorical_accuracy", "Accuracy")]):
    if key in h and f"val_{key}" in h:
        ax.plot(h[key], label=f"train {key}")
        ax.plot(h[f"val_{key}"], label=f"val {key}")
        ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle(f"MobileFaceNet FINE-TUNE on Bollywood Faces (best val_loss={hist_data['best_val_loss']:.4f})")
fig.tight_layout()
PLOT_PATH = f"{REPORTS_OUT}/mobilefacenet_ft_training_curves.png"
fig.savefig(PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot saved -> {PLOT_PATH}")

print("\n" + "=" * 78)
print(f"Final summary — Notebook 04a ({STRATEGY})")
print("=" * 78)
print(f"Backbone load method:    {load_method}")
print(f"Total celebs (classes):  {NUM_CLASSES}")
print(f"Train samples:           {len(train_files):,}")
print(f"Val samples:             {len(val_files):,}")
print(f"Epochs ran:              {hist_data['epochs_actual']} of {EPOCHS_MAX} max")
print(f"Final train acc:         {h['sparse_categorical_accuracy'][-1]:.4f}")
print(f"Final val   acc:         {h['val_sparse_categorical_accuracy'][-1]:.4f}")
print(f"Best  val   loss:        {hist_data['best_val_loss']:.4f}")
print()
print(f"TFLite file:             {TFLITE_PATH}")
print(f"TFLite size:             {os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB")
print(f"History JSON:            {HIST_PATH}")
print(f"Curves PNG:              {PLOT_PATH}")

## What to send back to Claude

Paste the literal output of these cells (in order):

1. **Cell 7** (discovery + split folder listing)
2. **Cell 9** (celeb gathering counts)
3. **Cell 11** (cropping detection rate)
4. **Cell 13** (train/val split sizes)
5. **Cell 15** (data pipeline smoke test + GPU check)
6. **Cell 17** (backbone load — `load_method`, params, sanity forward)
7. **Cell 19** (training model summary)
8. **Cell 21** (per-epoch training output)
9. **Cell 23** (TFLite conversion + shape check)
10. **Cell 25** (sanity cosine distances — same-celeb vs cross-celeb)
11. **Cell 27** (final summary)

If any cell fails, paste the **full traceback**. Especially Cell 17 — if both load strategies fail, switch this Kaggle account to running Notebook 04b instead and proceed there.

## After the run — committing the artifacts

Download to laptop into `kaggle_downloads/04a_mobilefacenet_ft/`:

- `/kaggle/working/models/mobilefacenet_bollywood_ft.tflite`
- `/kaggle/working/reports/mobilefacenet_ft_training_history.json`
- `/kaggle/working/reports/mobilefacenet_ft_training_curves.png`

Tell Claude when both 04a and 04b artifacts are downloaded. **Don't replace `mobile_app/assets/models/mobilefacenet.tflite` yet** — Notebook 05 evaluates both fine-tuned variants against the InsightFace baseline via pair verification, picks the EER winner, and only then do we ship.